# 02 Fine-tune YOLO11n on the campus classes

**Compute bucket: Colab/Kaggle GPU. About 30 to 60 minutes on a T4.**

Input: `MyDrive/reunite/reunite_dataset.zip` from notebook `01`. Output: `MyDrive/reunite/detector.pt` and
`detector_metrics.json`. Copy `detector.pt` to `backend/weights/` (or run `training/export.py`): the backend loads it automatically and
skips the COCO fallback.

In [ ]:
!pip -q install ultralytics

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import json, zipfile
from pathlib import Path

CLASSES = [
    "id_card", "laptop", "phone", "tablet", "earbuds", "headphones", "charger", "power_bank", "wallet",
    "keys", "backpack", "bottle", "book", "notebook", "spectacles", "watch", "umbrella", "calculator",
]  # same order and names as backend/app/ml/text/lexicons/categories.yaml
DEST = Path("/content/drive/MyDrive/reunite")
DATA = Path("/content/dataset")
if not DATA.exists():
    with zipfile.ZipFile(DEST / "reunite_dataset.zip") as z:
        z.extractall(DATA)
print(sorted(p.name for p in DATA.iterdir()))

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11n.pt")            # small on purpose: it must run on an 8 GB laptop
results = model.train(
    data=str(DATA / "dataset.yaml"), epochs=60, imgsz=640, batch=32, patience=15, seed=0,
    project="/content/runs", name="reunite", device=0, workers=2, cos_lr=True, close_mosaic=10,
)

In [ ]:
best = YOLO("/content/runs/reunite/weights/best.pt")
metrics = best.val(data=str(DATA / "dataset.yaml"))
per_class = {CLASSES[int(c)]: float(ap) for c, ap in zip(metrics.box.ap_class_index, metrics.box.ap50)}
report = {
    "map50": float(metrics.box.map50), "map50_95": float(metrics.box.map),
    "per_class_ap50": per_class, "detector_classes": {i: c for i, c in enumerate(CLASSES)},
    "weak_classes": [c for c in CLASSES if per_class.get(c, 0) < 0.5],
}
print(json.dumps(report, indent=2))

In [ ]:
import shutil
shutil.copy("/content/runs/reunite/weights/best.pt", DEST / "detector.pt")
json.dump(report, open(DEST / "detector_metrics.json", "w"), indent=2)
print("saved detector.pt and detector_metrics.json to", DEST)

Classes under 0.5 AP50 need more photos of that class (back to notebook `01`, add to `own/`). The backend's CLIP zero-shot path
still covers any class this detector is weak on, because the category head (notebook `03`) reads the whole crop, not the box.